# SEM Particle Detection

Workflow for extracting particle (x, y) positions from SEM images.

Steps:
1. Load and optionally invert the SEM image
2. Binarize using Sauvola local thresholding
3. Filter regions by area, shape, and solidity
4. Visualize detected particles overlaid on the image
5. (Optional) Analyse orientation and local ordering
6. Save particle positions to CSV

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from skimage.morphology import remove_small_holes, remove_small_objects

from phd_tools.sem import (
    fetch_file,
    binary_conversion,
    area_selection,
    visualise_selected_particules,
    save_positions_array,
    detect_angle_orientation,
    detect_angle_variation,
    detect_distance_variation,
)

## 1. Load image

Set `image_folder` and `image_file` to point to your SEM image.
Pass `invert=True` if your particles are dark on a bright background.

In [ ]:
image_folder = '../data/SEM_images'   # update this path
image_file   = 'PSi1406N1D200n_gravure_15062021_10.jpg'  # update this filename

img = fetch_file(image_folder, image_file, invert=False)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 8))
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.title('Raw SEM image')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '../data/PSi1406N1D200n_gravure_15062021_10.jpg'

## 2. Binarize

Sauvola local thresholding adapts to local contrast variations across the image.
Adjust `treshold_adjustment`, `window_size`, and `sigma_weight` to tune the segmentation.

In [ ]:
binary = binary_conversion(
    img,
    treshold_adjustment=0.09,
    window_size=41,
    delta_blur=2,
    sigma_weight=0.2,
)

<cell_type>markdown</cell_type>## 3. Morphological cleanup and region filtering

First apply morphological cleanup to the binary image:
- `remove_small_holes` fills small holes inside detected particles
- `remove_small_objects` removes small noise specks

Then filter regions by area, shape, and solidity:
- Regions smaller than `min_area` or larger than `max_area` pixels are discarded
- Elongated regions (major axis > 1.5 × equivalent diameter) and hollow regions are also removed

Returns:
- `particles` — Nx3 array (x, y, diameter)
- `final_areas` — binary mask of kept regions
- `props` — full regionprops table for inspection

In [ ]:
binary_clean = remove_small_holes(binary, area_threshold=2000)
binary_clean = remove_small_objects(binary_clean, min_size=100)

min_area = 10     # pixels — adjust to your particle size
max_area = 1500   # pixels — adjust to your particle size

particles, final_areas, props = area_selection(binary_clean, min_area=min_area, max_area=max_area)
print(f'Detected {len(particles)} particles')

## 4. Visualize detected particles

In [ ]:
visualise_selected_particules(particles, img)

## 5. (Optional) Orientation and ordering analysis

`detect_angle_orientation` computes the nearest-neighbour direction for each particle (0–60° mapped for hexagonal symmetry).
`detect_angle_variation` keeps only particles whose neighbours share a similar orientation (locally ordered).
`detect_distance_variation` keeps only particles with a regular nearest-neighbour distance.

In [ ]:
positions = particles[:, :2]  # use only x, y columns

particles_direction, angles = detect_angle_orientation(positions, label=None)
print(f'Angle range: {angles.min():.1f}° – {angles.max():.1f}°')

# Keep particles with locally consistent orientation (threshold in degrees)
ordered_by_angle = detect_angle_variation(positions, angles, treshold=5)
print(f'Ordered by angle: {len(ordered_by_angle)} particles')

# Keep particles with locally consistent spacing
ordered_by_distance = detect_distance_variation(positions)
print(f'Ordered by distance: {len(ordered_by_distance)} particles')

## 6. Save particle positions

`norm_factor` converts pixel coordinates to physical units (e.g. 1e-9 for nm→m).

In [ ]:
output_folder   = '../data'
output_filename = 'particles_detected'   # saved as particles_detected.csv
norm_factor     = 1e-9                   # pixel → metres; set to 1 to keep pixels

save_positions_array(output_filename, output_folder, particles, norm_factor)
print(f'Saved to {output_folder}/{output_filename}.csv')